In [116]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os
import joblib

# feature_path = os.path.abspath(r"C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor")
feature_path = os.path.abspath('/Users/alexg/Documents/Documents/Prize-Picks-Prop-Predictor')
if feature_path not in sys.path:
    sys.path.append(feature_path)

# from FEATURE_ENGINEERING.features import *
from MODELS.model import *
from MODELS.pipeline import *
pd.set_option('display.max_columns', None)

In [35]:
features = [
    # Player context
    'PLAYER_ID', 'TEAM_ID', 'OPP_TEAM_ID', 
    'STARTING', 'HOME_GAME', 
    'PLAYER_DAYS_REST', 'IS_BACK_TO_BACK', 
    
    # Star Players
    # 'PLAYER_IS_TEAM_STAR', 'TEAM_STAR_OUT',
    # 'PTS_WITHOUT_STAR', 'MIN_WITHOUT_STAR', 'USG_PCT_WITHOUT_STAR', 'FGA_WITHOUT_STAR', 'FG3A_WITHOUT_STAR', 'FTA_WITHOUT_STAR', 
    # 'EFG_PCT_WITHOUT_STAR', 'TS_PCT_WITHOUT_STAR', 'AST_WITHOUT_STAR', 'REB_WITHOUT_STAR', 'PTS_PER_36_WITHOUT_STAR',
    
    # Player season averages
    'MIN_SEASON_AVG_TO_DATE', 'PTS_SEASON_AVG_TO_DATE','FGA_SEASON_AVG_TO_DATE','FG3A_SEASON_AVG_TO_DATE',
    'FTA_SEASON_AVG_TO_DATE','USG_PCT_SEASON_AVG_TO_DATE','TS_PCT_SEASON_AVG_TO_DATE',
    'EFG_PCT_SEASON_AVG_TO_DATE', 'AST_SEASON_AVG_TO_DATE', 'REB_SEASON_AVG_TO_DATE', 'TOV_SEASON_AVG_TO_DATE',
    
    # LAG
    'PTS_LAG_1', 'PTS_LAG_2',
    'FGA_LAG_1', 'FGA_LAG_2',
    'MIN_LAG_1', 'MIN_LAG_2',
    'USG_PCT_LAG_1', 'USG_PCT_LAG_2',
    
    # Short-term form (5-game rolling averages)
    'MIN_ROLLING_AVG_5', 'PTS_ROLLING_AVG_5', 'FGA_ROLLING_AVG_5',
    'FG3A_ROLLING_AVG_5', 'FTA_ROLLING_AVG_5', 'USG_PCT_ROLLING_AVG_5',
    'TS_PCT_ROLLING_AVG_5','EFG_PCT_ROLLING_AVG_5', 'AST_ROLLING_AVG_5', 
    'REB_ROLLING_AVG_5', 'TOV_ROLLING_AVG_5',
    
    # Medium-term form (15-game rolling averages)
    'MIN_ROLLING_AVG_15', 'PTS_ROLLING_AVG_15', 'FGA_ROLLING_AVG_15',
    'FG3A_ROLLING_AVG_15', 'FTA_ROLLING_AVG_15', 'USG_PCT_ROLLING_AVG_15',
    'TS_PCT_ROLLING_AVG_15','EFG_PCT_ROLLING_AVG_15', 'AST_ROLLING_AVG_15', 
    'REB_ROLLING_AVG_15', 'TOV_ROLLING_AVG_15',
    
    # Long-term form (40-game rolling averages)
    'MIN_ROLLING_AVG_40', 'PTS_ROLLING_AVG_40', 'FGA_ROLLING_AVG_40', 'FG3A_ROLLING_AVG_40', 'FTA_ROLLING_AVG_40',
    'USG_PCT_ROLLING_AVG_40', 'TS_PCT_ROLLING_AVG_40', 'EFG_PCT_ROLLING_AVG_40', 'AST_ROLLING_AVG_40', 
    'REB_ROLLING_AVG_40', 'TOV_ROLLING_AVG_40',

    # Opponent
    'OPP_DEF_RATING_AVG_TO_DATE', 'OPP_PACE_AVG_TO_DATE', 'OPP_PTS_AVG_TO_DATE', 'OPP_FGA_AVG_TO_DATE', 
    'OPP_REB_AVG_TO_DATE', 'OPP_AST_AVG_TO_DATE', 'OPP_TOV_AVG_TO_DATE', 'OPP_BLK_AVG_TO_DATE', 'OPP_STL_AVG_TO_DATE',
    
    #starters 
    'TEAM_OFF_RATING_AVG_TO_DATE','TEAM_DEF_RATING_AVG_TO_DATE','TEAM_PACE_AVG_TO_DATE', 'TEAM_FGA_AVG_TO_DATE',
    'TEAM_PTS_AVG_TO_DATE', 'TEAM_REB_AVG_TO_DATE', 'TEAM_AST_AVG_TO_DATE', 'TEAM_TOV_AVG_TO_DATE',

    # Matchup micro-feature
    'MATCHUP_AVG_MIN_LAST_3_TO_DATE', 'MATCHUP_AVG_FGA_LAST_3_TO_DATE', 'MATCHUP_AVG_FG3A_LAST_3_TO_DATE', 'MATCHUP_AVG_FTA_LAST_3_TO_DATE', 
    'MATCHUP_AVG_PTS_LAST_3_TO_DATE', 'MATCHUP_AVG_USG_PCT_LAST_3_TO_DATE', 'MATCHUP_AVG_EFG_PCT_LAST_3_TO_DATE', 'MATCHUP_AVG_TS_PCT_LAST_3_TO_DATE', 
    'MATCHUP_AVG_AST_LAST_3_TO_DATE', 'MATCHUP_AVG_REB_LAST_3_TO_DATE', 'MATCHUP_AVG_TOV_LAST_3_TO_DATE',
    
    # Team odds
    'team_spread', 'total', 'team_is_favored','TEAM_IMPLIED_PTS_FAV','TEAM_IMPLIED_PTS_UND','BLOWOUT_RISK'
]


In [194]:
data = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_25.csv')
odds = pd.read_csv('../BACKTESTS/odds25.csv')
model = joblib.load('/Users/alexg/Documents/Documents/Prize-Picks-Prop-Predictor/MODELS/Models/PTS_cat_model.pkl')
starters = getStarters(22400307, 'CLE', data)
games = get_espn_games(20250411)

/var/folders/nw/9w4r5hrd05s122kt5hg_t8bw0000gn/T/ipykernel_39193/3939896572.py:2: DtypeWarning: Columns (10,11,13) have mixed types. Specify dtype option on import or set low_memory=False.
  odds = pd.read_csv('../BACKTESTS/odds25.csv')


In [42]:


def makePredictionCatBoost(player_name, data, model, bookmakers, games, todayDate, starters, game_id, features, n_games=3):
    from catboost import Pool
    import pandas as pd
    
    # Get feature vector
    feature_vector = buildFeatureVector(player_name, data, games, todayDate, starters, game_id, n_games=n_games)
    
    # Convert to DataFrame with proper feature names
    X = pd.DataFrame([feature_vector], columns=features)
    
    # Define categorical features (same as used during training)
    categorical_cols = ['PLAYER_ID', 'TEAM_ID', 'OPP_TEAM_ID']
    cat_cols = [c for c in categorical_cols if c in features]
    cat_idx = [features.index(c) for c in cat_cols]
    
    # Data type cleanup (matching training preprocessing)
    for c in X.columns:
        if c not in cat_cols:
            if X[c].dtype == 'bool':
                X[c] = X[c].astype(int)
            elif X[c].dtype == 'object':
                X[c] = pd.to_numeric(X[c], errors='coerce')
    
    # Create CatBoost Pool with categorical features
    pool = Pool(X, cat_features=cat_idx)
    
    # Make prediction
    prediction = model.predict(pool)[0]
    
    prop_line = bookmakers[bookmakers['player'] == player_name]['line'].values[0]
    
    # Get opponent team for display
    player_id, player_team = findPlayerID(player_name, data)
    opponent, homeGame = findOppTeam(player_name, data, games)
    
    return {
        'player': player_name,
        'opponent': opponent,
        'predicted_stat': round(prediction, 2),
        'raw_prediction': prediction,
        'prop_line': prop_line,
        'edge': round(prediction - prop_line, 2),
        'recommendation': 'OVER' if prediction > prop_line else 'UNDER'
    }

pred = makePredictionCatBoost('LeBron James', data, model, odds, games, 20250411, starters, game_id=22401185, features=features)
pred

{'player': 'LeBron James',
 'opponent': 'HOU',
 'predicted_stat': np.float64(19.91),
 'raw_prediction': np.float64(19.914724424158344),
 'prop_line': np.float64(23.5),
 'edge': np.float64(-3.59),
 'recommendation': 'UNDER'}

In [201]:
s25 = pd.read_csv('../DATA/CSV_FILES/REGULAR_DATA/s25.csv')
odds = pd.read_csv('../BACKTESTS/odds25.csv')
odds.drop(columns=['Unnamed: 0', 'Unnamed: 0.1'], inplace=True)
odds

/var/folders/nw/9w4r5hrd05s122kt5hg_t8bw0000gn/T/ipykernel_39193/1220797406.py:2: DtypeWarning: Columns (10,11,13) have mixed types. Specify dtype option on import or set low_memory=False.
  odds = pd.read_csv('../BACKTESTS/odds25.csv')


,NAME,CATEGORY,BOOKMAKER,OVER/UNDER,LINE,PRICE,HOME_TEAM,AWAY_TEAM,game_id,commence_time,GAME_DATE,period_id,fair_line,fair_odds
0,Miles McBride,player_points,prizepicks,Under,8.5,-137,Boston Celtics,New York Knicks,92cf7db605a734b10e69eabf56e1eac9,2024-10-22T23:40:00Z,2024-10-22,NaN,NaN,NaN
1,OG Anunoby,player_assists,prizepicks,Over,1.5,-137,Boston Celtics,New York Knicks,92cf7db605a734b10e69eabf56e1eac9,2024-10-22T23:40:00Z,2024-10-22,NaN,NaN,NaN
2,OG Anunoby,player_assists,prizepicks,Under,1.5,-137,Boston Celtics,New York Knicks,92cf7db605a734b10e69eabf56e1eac9,2024-10-22T23:40:00Z,2024-10-22,NaN,NaN,NaN
3,Jayson Tatum,player_points,prizepicks,Over,26.5,-137,Boston Celtics,New York Knicks,92cf7db605a734b10e69eabf56e1eac9,2024-10-22T23:40:00Z,2024-10-22,NaN,NaN,NaN
4,Jayson Tatum,player_points,prizepicks,Under,26.5,-137,Boston Celtics,New York Knicks,92cf7db605a734b10e69eabf56e1eac9,2024-10-22T23:40:00Z,2024-10-22,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
198104,Harrison Barnes,rebounds,underdog,over,2.5,100,San Antonio Spurs,Toronto Raptors,NaN,NaN,2025-04-13,game,2.5,-115.0
198105,Harrison Barnes,rebounds,draftkings,over,2.5,-145,San Antonio Spurs,Toronto Raptors,NaN,NaN,2025-04-13,game,2.5,-115.0
198106,Harrison Barnes,rebounds,underdog,under,2.5,100,San Antonio Spurs,Toronto Raptors,NaN,NaN,2025-04-13,game,2.5,115.0
198107,Chris Paul,points,underdog,over,5.5,100,San Antonio Spurs,Toronto Raptors,NaN,NaN,2025-04-13,game,6.0,100.0
